# Phase-2 / Cell 1 (v2): Setup + CSV rebuild + DataLoaders + Load & Freeze Encoder (no-norm)

In [1]:
# Phase-2 / Cell 1 (v2):
# Setup + CSV rebuild + DataLoaders + Load & Freeze Encoder ============

!pip -q install timm==0.9.10

import os, json, shutil, torch, timm
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Phase-1 artifacts (classifier checkpoint etc.)
# SAME as before
PHASE1_DIR = "/kaggle/input/vit_classifier_cub_2011/pytorch/default/1"
assert os.path.isdir(PHASE1_DIR), f"Phase-1 dir not found: {PHASE1_DIR}"
assert os.path.isfile(os.path.join(PHASE1_DIR, "meta_cls.json")), "meta_cls.json missing in Phase-1 dir"
assert os.path.isfile(os.path.join(PHASE1_DIR, "vit_cls_best.pt")), "vit_cls_best.pt missing in Phase-1 dir"
print("Phase-1 dir:", PHASE1_DIR)

# Load Phase-1 meta (this tells us img_size, patch_size, mean/std etc.)
with open(os.path.join(PHASE1_DIR, "meta_cls.json")) as f:
    meta_cls = json.load(f)

IMG_SIZE    = int(meta_cls.get("img_size", 224))
PATCH_SIZE  = int(meta_cls.get("patch_size", 16))
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2
TIMM_NAME   = meta_cls.get("timm_name", "vit_base_patch16_224")

MEAN = meta_cls.get("mean", [0.485,0.456,0.406])
STD  = meta_cls.get("std",  [0.229,0.224,0.225])

print(f"img_size={IMG_SIZE}, patch_size={PATCH_SIZE}, num_patches={NUM_PATCHES}, timm_name={TIMM_NAME}")
print(f"mean={MEAN}, std={STD}")

# Working dir on Kaggle -----
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

# Build CSVs (train/val/test) if missing -----
need_build = False
for name in ["cub_train.csv","cub_val.csv","cub_test.csv"]:
    if not os.path.isfile(os.path.join(WORK, name)):
        need_build = True
        break

if need_build:
    RAW_DIR = "/kaggle/input/cub2002011/CUB_200_2011"
    IMG_DIR = os.path.join(RAW_DIR, "images")
    classes_file = os.path.join(RAW_DIR, "classes.txt")
    images_file  = os.path.join(RAW_DIR, "images.txt")
    labels_file  = os.path.join(RAW_DIR, "image_class_labels.txt")
    split_file   = os.path.join(RAW_DIR, "train_test_split.txt")

    # load meta files from CUB
    df_images = pd.read_csv(images_file, sep=" ", names=["img_id","rel_path"])
    df_labels = pd.read_csv(labels_file, sep=" ", names=["img_id","class_id"])  # 1..200
    df_split  = pd.read_csv(split_file,  sep=" ", names=["img_id","is_train"])  # 1=train, 0=test

    df = df_images.merge(df_labels, on="img_id").merge(df_split, on="img_id")
    df["class_id"] = df["class_id"] - 1                  # make it 0..199 instead of 1..200
    df["abs_path"] = df["rel_path"].apply(lambda x: os.path.join(IMG_DIR, x))
    df["split"]    = np.where(df["is_train"]==1, "train", "test")

    df_train_raw = df[df["split"]=="train"].copy()
    df_test      = df[df["split"]=="test"].copy()

    # stratified 15% val from train (SAME logic as old code)
    idx = np.arange(len(df_train_raw))
    train_idx, val_idx = train_test_split(
        idx,
        test_size=0.15,
        random_state=42,
        stratify=df_train_raw["class_id"].values
    )
    df_train = df_train_raw.iloc[train_idx].copy()
    df_val   = df_train_raw.iloc[val_idx].copy()

    df_train.to_csv(os.path.join(WORK, "cub_train.csv"), index=False)
    df_val.to_csv(  os.path.join(WORK, "cub_val.csv"),   index=False)
    df_test.to_csv( os.path.join(WORK, "cub_test.csv"),  index=False)

    print("Rebuilt CSVs (stratified 15% val).",
          "Train/Val/Test:", len(df_train), len(df_val), len(df_test))
else:
    print("CSVs already present in", WORK)

# Transforms for AE training
# NOTE: exactly like your original AE: NO Normalize().
# We keep pixels in [0,1] so decoder learns to spit out [0,1] RGB.
# (Later, in training cell, we will add perceptual loss etc.)
def make_tfm(size):
    return transforms.Compose([
        transforms.Resize(int(size*1.15)),
        transforms.CenterCrop(size),
        transforms.ToTensor(),        # -> [0,1]
    ])

train_tf = make_tfm(IMG_SIZE)
eval_tf  = make_tfm(IMG_SIZE)

class CUBAE(Dataset):
    def __init__(self, csv_path, tfm):
        df = pd.read_csv(csv_path)
        self.paths = df["abs_path"].tolist()
        self.tfm = tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        x = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(x)            # [3,H,W] in [0,1]

ds_train = CUBAE(os.path.join(WORK,"cub_train.csv"), train_tf)
ds_val   = CUBAE(os.path.join(WORK,"cub_val.csv"),   eval_tf)
ds_test  = CUBAE(os.path.join(WORK,"cub_test.csv"),  eval_tf)

# batch size rule from your code
BATCH = 32 if IMG_SIZE == 224 else 16

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False,
                      num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False,
                      num_workers=2, pin_memory=True)

xb = next(iter(dl_train))
print("one batch:", tuple(xb.shape), "range:", (float(xb.min()), float(xb.max())))

# Build ViT encoder (no head) & load Phase-1 weights; then FREEZE
# same idea as your vit_enc
vit_enc = timm.create_model(TIMM_NAME, pretrained=False, num_classes=0)  # encoder-only
state = torch.load(os.path.join(PHASE1_DIR, "vit_cls_best.pt"), map_location="cpu")

# Allow missing classifier head keys etc.
missing, unexpected = vit_enc.load_state_dict(state, strict=False)
print("load_state_dict -> missing:", missing, "| unexpected:", unexpected)

vit_enc.to(DEVICE).eval()
for p in vit_enc.parameters():
    p.requires_grad = False          # Stage-1: encoder frozen

# quick forward to confirm token shape using vit_enc(x) direct
with torch.no_grad():
    toks_direct = vit_enc(xb.to(DEVICE))
print("encoder tokens via vit_enc(x):", tuple(toks_direct.shape))
# e.g. [B, 197, 768] for ViT-B/16 @224

# count trainable encoder params (should be 0 now)
trainable = sum(p.numel() for p in vit_enc.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vit_enc.parameters())
print(f"encoder params: total={total:,} | trainable={trainable:,} (frozen for Stage-1)")

# ----- Helper vit_get_tokens() EXACTLY like before -----
import torch

@torch.no_grad()
def vit_get_tokens(model, x):
    """
    Safe forward pass to grab ALL tokens [CLS + patches]
    from timm ViT-B/16, consistent with our Phase-1 encoder.
    x: [B,3,H,W] in [0,1]  (note: not normalized here)
    return: [B, 1+N_patches, C]  e.g. [B,197,768]
    """
    B = x.size(0)
    # timm ViT forward pieces:
    xx = model.patch_embed(x)                     # [B, N, C]
    cls_tok = model.cls_token.expand(B, -1, -1)   # [B, 1, C]
    if getattr(model, 'pos_embed', None) is not None:
        xx = torch.cat((cls_tok, xx), dim=1)      # [B, 1+N, C]
        xx = xx + model.pos_embed
    xx = model.pos_drop(xx)
    for blk in model.blocks:
        xx = blk(xx)
    xx = model.norm(xx)                           # [B, 1+N, C]
    return xx

# sanity check tokens from vit_get_tokens
with torch.no_grad():
    toks_full = vit_get_tokens(vit_enc, xb.to(DEVICE))
print("vit_get_tokens shape:", tuple(toks_full.shape))  # expect (B, 1+N, 768) ~ (32,197,768) for 224

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 25.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 47.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Phase-2 / Cell 2 (v2): Strong decoder + perceptual loss + long training

In [ ]:
# Phase-2 / Cell 2 (v2):
# Strong decoder + perceptual loss + long training

!pip -q install lpips==0.1.4
import math, time, json, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import lpips  # perceptual loss


# Config / hyperparams (UPGRADED)
EMBED_DIM   = vit_enc.num_features      # should be 768 for ViT-B/16
DEC_DIM     = 768                      # was 512 -> now 768 to match encoder width
DEC_DEPTH   = 8                        # was 4 -> now 8 (more capacity; you can try 12 later)
DEC_HEADS   = 12                       # was 8 -> now 12 (standard for ViT-B/16 width)
PATCH       = PATCH_SIZE               # same as before (16)
IMG         = IMG_SIZE                 # 224
OUT_CH      = 3                        # RGB
N_PATCHES   = (IMG // PATCH) ** 2      # should be 196 for 224/16

EPOCHS      = 150                      # was 100 -> now 200 for better recon
WARMUP      = 5
LR          = 1e-4
WD          = 5e-2
PATIENCE    = 30                       # give it more patience because longer training

# loss weights
W_MSE       = 0.7
W_LPIPS     = 0.3

WORK = "/kaggle/working"
os.makedirs(os.path.join(WORK, "recon_samples_v2"), exist_ok=True)


# Helper: save comparison grid (orig vs recon) for visual sanity check

def save_grid(orig, recon, path, max_n=6):
    # orig, recon are [B,3,H,W] in [0,1]
    B = min(max_n, orig.size(0))
    fig, axes = plt.subplots(B, 2, figsize=(4, 2*B), dpi=120)
    for i in range(B):
        for j, img in enumerate([orig[i], recon[i]]):
            axes[i, j].imshow(img.permute(1,2,0).detach().cpu().numpy())
            axes[i, j].axis('off')
            axes[i, j].set_title('orig' if j==0 else 'recon')
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.close(fig)


# Stronger decoder (v2)

class StrongDecoder(nn.Module):
    """
    This is the upgraded version of TinyMAEDecoder:
    - wider dim (=768 instead of 512),
    - deeper transformer stack (=8 instead of 4),
    - more heads (=12 instead of 8).
    """

    def __init__(self,
                 in_dim=EMBED_DIM,
                 dec_dim=DEC_DIM,
                 depth=DEC_DEPTH,
                 heads=DEC_HEADS,
                 num_patches=N_PATCHES,
                 patch=PATCH,
                 out_ch=OUT_CH):
        super().__init__()
        self.num_patches = num_patches
        self.patch = patch
        self.out_ch = out_ch

        # project encoder tokens -> decoder space
        self.proj_in = nn.Linear(in_dim, dec_dim)

        # build TransformerEncoder with <depth> layers
        layer = nn.TransformerEncoderLayer(
            d_model=dec_dim,
            nhead=heads,
            dim_feedforward=dec_dim * 4,
            dropout=0.0,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)

        self.norm = nn.LayerNorm(dec_dim)

        # map each decoded patch token -> flattened patch pixels (3 * P * P)
        self.head = nn.Linear(dec_dim, out_ch * patch * patch)

    def forward(self, tokens_all):
        # tokens_all: [B, 1+N, C]  (CLS + patch tokens)
        # drop CLS, only decode patch tokens
        x = tokens_all[:, 1:, :]          # [B, N, C]
        x = self.proj_in(x)               # [B, N, dec_dim]
        x = self.blocks(x)                # [B, N, dec_dim]
        x = self.norm(x)                  # [B, N, dec_dim]

        # final linear to patch pixels
        x = self.head(x)                  # [B, N, out_ch*P*P]

        B, N, PP = x.shape
        P = self.patch                   # e.g.16
        # reshape tokens -> image
        x = x.view(B, N, self.out_ch, P, P)      # [B, N, 3, P, P]
        h = w = int(math.sqrt(N))                # should be 14x14 for 196 patches
        x = x.view(B, h, w, self.out_ch, P, P)   # [B,14,14,3,16,16] for 224
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()  # [B,3,14,16,14,16]
        x = x.view(B, self.out_ch, h*P, w*P)     # [B,3,224,224]

        # squash to [0,1]
        x = torch.sigmoid(x)
        return x

mae_dec = StrongDecoder().to(DEVICE)


# Optimizer / Scheduler / AMP scaler
# Only decoder is trainable (encoder stays frozen)

optimizer = AdamW(mae_dec.parameters(), lr=LR, weight_decay=WD)

def lr_lambda(epoch):
    # same cosine schedule with warmup as before
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = (epoch - WARMUP) / float(max(1, EPOCHS - WARMUP))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler(enabled=(DEVICE == "cuda"))

# base MSE reconstruction loss
mse_criterion = nn.MSELoss()

# perceptual LPIPS model (no training, eval mode)
lpips_model = lpips.LPIPS(net='vgg').to(DEVICE)
lpips_model.eval()


# Single epoch pass

def run_epoch(loader, train=True):
    if train:
        mae_dec.train()
    else:
        mae_dec.eval()

    total_loss = 0.0
    total_mse  = 0.0
    total_lp   = 0.0
    total_count = 0

    for xb in loader:
        xb = xb.to(DEVICE, non_blocking=True)   # [B,3,224,224] in [0,1]

        with torch.no_grad():
            toks = vit_get_tokens(vit_enc, xb)  # [B,1+N,768], encoder frozen

        with torch.set_grad_enabled(train):
            with autocast(enabled=(DEVICE=="cuda")):
                recon = mae_dec(toks)           # [B,3,224,224] in [0,1]

                # Loss 1: pixel MSE
                mse_loss = mse_criterion(recon, xb)

                # Loss 2: perceptual LPIPS
                # LPIPS expects inputs in [-1,1]
                recon_lp = recon * 2.0 - 1.0
                xb_lp    = xb    * 2.0 - 1.0
                lp = lpips_model(recon_lp, xb_lp).mean()

                loss = W_MSE * mse_loss + W_LPIPS * lp

        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = xb.size(0)
        total_loss  += loss.detach().item()     * bs
        total_mse   += mse_loss.detach().item() * bs
        total_lp    += lp.detach().item()       * bs
        total_count += bs

    avg_total = total_loss / max(1,total_count)
    avg_mse   = total_mse  / max(1,total_count)
    avg_lp    = total_lp   / max(1,total_count)
    return avg_total, avg_mse, avg_lp


# Training loop

best_val = float('inf')
no_improve = 0
log_v2 = []

print(f"Training AE v2 (encoder frozen) for {EPOCHS} epochs…")

for ep in range(1, EPOCHS+1):
    tr_total, tr_mse, tr_lp = run_epoch(dl_train, train=True)
    va_total, va_mse, va_lp = run_epoch(dl_val,   train=False)

    scheduler.step()

    log_v2.append({
        "epoch": ep,
        "train_total": tr_total,
        "train_mse": tr_mse,
        "train_lpips": tr_lp,
        "val_total": va_total,
        "val_mse": va_mse,
        "val_lpips": va_lp,
        "lr": optimizer.param_groups[0]['lr'],
    })

    print(f"[{ep:03d}/{EPOCHS}] "
          f"train total {tr_total:.5f} | mse {tr_mse:.5f} | lp {tr_lp:.5f}  ||  "
          f"val total {va_total:.5f} | mse {va_mse:.5f} | lp {va_lp:.5f}")

    # preview recon every 5 epochs
    if ep % 5 == 0:
        with torch.no_grad():
            xb_vis = next(iter(dl_val)).to(DEVICE)
            toks_vis = vit_get_tokens(vit_enc, xb_vis)
            rec_vis  = mae_dec(toks_vis)  # [0,1]
        grid_path = os.path.join(WORK, "recon_samples_v2", f"epoch_{ep:03d}.png")
        save_grid(xb_vis, rec_vis, grid_path)
        print("  ↳ saved preview:", grid_path)

    # best checkpoint logic (using val_total)
    if va_total < best_val:
        best_val = va_total
        no_improve = 0
        torch.save(mae_dec.state_dict(), os.path.join(WORK, "vit_ae_decoder_v2.pt"))
        torch.save(vit_enc.state_dict(), os.path.join(WORK, "vit_ae_encoder_v2.pt"))
        print("  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping: no val improvement.")
            break


# Save logs + meta_ae_v2.json

import pandas as pd

df_log_v2 = pd.DataFrame(log_v2)
df_log_v2.to_csv(os.path.join(WORK, "ae_train_log_v2.csv"), index=False)

meta_ae_v2 = {
    "encoder_timm_name": TIMM_NAME,
    "img_size": IMG,
    "patch_size": PATCH,
    "num_patches": N_PATCHES,
    "embed_dim": EMBED_DIM,
    "decoder_dim": DEC_DIM,        # 768 now
    "decoder_depth": DEC_DEPTH,    # 8 now
    "decoder_heads": DEC_HEADS,    # 12 now
    "out_channels": OUT_CH,
    "loss_weights": {"mse": W_MSE, "lpips": W_LPIPS},
    "epochs_trained": ep,
}
with open(os.path.join(WORK, "meta_ae_v2.json"), "w") as f:
    json.dump(meta_ae_v2, f, indent=2)

print("Saved:",
      ["vit_ae_decoder_v2.pt",
       "vit_ae_encoder_v2.pt",
       "meta_ae_v2.json",
       "ae_train_log_v2.csv",
       "recon_samples_v2/*"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.4 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipykernel_38/561226750.py:140: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE == "cuda"))
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the m

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 225MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth
Training AE v2 (encoder frozen) for 150 epochs…


/tmp/ipykernel_38/561226750.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


[001/150] train total 0.23953 | mse 0.06343 | lp 0.65044  ||  val total 0.22410 | mse 0.05710 | lp 0.61381
  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt
[002/150] train total 0.21799 | mse 0.05612 | lp 0.59569  ||  val total 0.21135 | mse 0.05312 | lp 0.58055
  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt
[003/150] train total 0.20499 | mse 0.04576 | lp 0.57655  ||  val total 0.19775 | mse 0.03790 | lp 0.57074
  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt
[004/150] train total 0.18908 | mse 0.03118 | lp 0.55750  ||  val total 0.18047 | mse 0.02636 | lp 0.54007
  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt
[005/150] train total 0.17127 | mse 0.02439 | lp 0.51398  ||  val total 0.16784 | mse 0.02308 | lp 0.50561
  ↳ saved preview: /kaggle/working/recon_samples_v2/epoch_005.png
  ↳ saved vit_ae_decoder_v2.pt and vit_ae_encoder_v2.pt
[006/150] train total 0.16233 | mse 0.02205 | lp 0.48966  ||  val total 0.16154 | mse 0.02192 | lp 0.48730
  ↳ saved vi

# Phase-2 / Cell 3 (v2): Pack best checkpoint + logs + previews into a single ZIP for download

In [ ]:
# Phase-2 / Cell 3 (v2):
# Pack best checkpoint + logs + previews into a single ZIP for download ============

import os, shutil, glob, json

OUT = "/kaggle/working"

phase2_dir = os.path.join(OUT, "phase2_out_v2")
os.makedirs(phase2_dir, exist_ok=True)

# --- which files we want to export for MC/MI notebook ---
want = [
    "vit_ae_encoder_v2.pt",   # frozen ViT encoder (same arch as classifier backbone)
    "vit_ae_decoder_v2.pt",   # our new StrongDecoder weights after training
    "meta_ae_v2.json",        # config for this v2 model (dim=768, depth=8, heads=12, etc.)
    "ae_train_log_v2.csv",    # training log with total/mse/lpips curves
]

copied = []
for name in want:
    src = os.path.join(OUT, name)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(phase2_dir, name))
        copied.append(name)
    else:
        print(f"[warn] missing {name}")

# include recon sample grids from training
samples_dir = os.path.join(OUT, "recon_samples_v2")
if os.path.isdir(samples_dir):
    dst_samples = os.path.join(phase2_dir, "recon_samples_v2")
    os.makedirs(dst_samples, exist_ok=True)

    # grab all epoch_*.png previews we saved
    for p in sorted(glob.glob(os.path.join(samples_dir, "epoch_*.png"))):
        shutil.copy2(p, os.path.join(dst_samples, os.path.basename(p)))

# write a README to explain how to use in MC/MI notebook
readme_path = os.path.join(phase2_dir, "README_v2.txt")
with open(readme_path, "w") as f:
    f.write(
"""ViT Autoencoder v2 (encoder frozen during training)

WHAT THIS IS:
- vit_ae_encoder_v2.pt  : ViT-B/16 encoder state_dict. This matches your classifier backbone.
- vit_ae_decoder_v2.pt  : StrongDecoder state_dict.
  StrongDecoder = dim=768, depth=8 transformer decoder, heads=12, trained ~50 epochs.
  Loss = 0.7*MSE + 0.3*LPIPS (VGG-based perceptual). Much sharper than v1.
- meta_ae_v2.json       : JSON config describing image size, patch size, heads, depth, etc.
- ae_train_log_v2.csv   : training curve (train_total, val_total, mse, lpips).

HOW TO USE IN YOUR MC/MI / COUNTERFACTUAL NOTEBOOK:
1. Rebuild the same ViT encoder arch (timm.create_model(..., num_classes=0)).
2. Load state_dict from vit_ae_encoder_v2.pt.
   Make sure requires_grad=False when you generate explanations.
3. Define StrongDecoder(...) EXACTLY like in Cell 2 (v2) of training notebook
   (same dec_dim=768, dec_depth=8, dec_heads=12, patch=16, etc.)
4. Load state_dict from vit_ae_decoder_v2.pt into that StrongDecoder.
5. Get tokens:
      toks = vit_get_tokens(vit_enc, img_batch[0:1])
   (img in [0,1], no normalization)
6. For ORIGINAL recon:
      recon_orig = decoder(toks)
7. For COUNTERFACTUAL recon:
      - apply MI boost (your flip pipeline) to toks first
      - then pass boosted toks into decoder
      recon_cf = decoder(toks_boosted)
8. Show side-by-side:
      recon_orig vs recon_cf
   and diff heatmap (recon_cf - recon_orig or your saliency map logic).

GOAL:
Use these reconstructions + flip evidence text block as thesis figure.
"""
    )

# finally ZIP everything so you can download one file
zip_base = os.path.join(OUT, "ViT_Autoencoder_CUB200_v2")
shutil.make_archive(zip_base, "zip", phase2_dir)

print("ZIP created:", zip_base + ".zip")
print("\nContents in phase2_out_v2/:")
for p in sorted(glob.glob(os.path.join(phase2_dir, "**/*"), recursive=True)):
    print("-", os.path.relpath(p, phase2_dir))
